# ICP-OES / ICP-MS Elemental Analysis & Catalyst Leaching

This notebook demonstrates quantitative analytical chemistry processing:
- Multi-point standard calibration with weighted linear regression
- IUPAC Limit of Detection ($\mathrm{LOD} = 3.3\,\sigma_{\mathrm{blank}} / S$) and Limit of Quantification ($\mathrm{LOQ} = 10\,\sigma_{\mathrm{blank}} / S$)
- Solvent blank correction and sample dilution adjustment
- Quantitative catalyst metal leaching dissolution loss (% and $\mu\text{g/mg}_{\text{cat}}$).

In [1]:
import sys
from pathlib import Path
root_path = Path.cwd().parent.parent if Path.cwd().name == 'characterization' else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from parsers.characterization import parse_icp_file
from modules.characterization.icp import (
    fit_icp_calibration,
    calculate_catalyst_leaching,
    plot_icp_calibration,
    plot_icp_leaching_comparison,
)
from scripts.utils import apply_publication_style

apply_publication_style()
print('ICP modules loaded.')

ICP modules loaded.


## 1. Multi-Point ICP Calibration & LOD Determination
Evaluate calibration standards for Pt, Ni, Fe.

In [2]:
cal_path = root_path / 'data' / 'examples' / 'characterization' / 'icp' / 'icp_calibration_standards.csv'
cal_data = parse_icp_file(cal_path)
df_cal = cal_data.dataframe

for elem in ['Pt', 'Ni', 'Fe']:
    sub = df_cal[df_cal['element'] == elem]
    stds = sub[sub['sample_id'].str.startswith('STD')]
    blanks = sub[sub['sample_id'].str.startswith('BLANK')]
    res = fit_icp_calibration(stds['concentration'].values, stds['intensity'].values, blank_intensities=blanks['intensity'].values)
    print(f'{elem}: Slope = {res["slope"]:.1f} cts/(mg/L), R^2 = {res["r_squared"]:.5f}, LOD = {res["lod_conc"]:.3f} mg/L, LOQ = {res["loq_conc"]:.3f} mg/L')

# Plot Ni calibration curve
sub_ni = df_cal[df_cal['element'] == 'Ni']
stds_ni = sub_ni[sub_ni['sample_id'].str.startswith('STD')]
blanks_ni = sub_ni[sub_ni['sample_id'].str.startswith('BLANK')]
cal_ni = fit_icp_calibration(stds_ni['concentration'].values, stds_ni['intensity'].values, blank_intensities=blanks_ni['intensity'].values)
fig, axes = plot_icp_calibration(cal_ni, element='Nickel')
plt.show()

Pt: Slope = 12504.1 cts/(mg/L), R^2 = 1.00000, LOD = 0.002 mg/L, LOQ = 0.007 mg/L
Ni: Slope = 28402.2 cts/(mg/L), R^2 = 1.00000, LOD = 0.001 mg/L, LOQ = 0.002 mg/L
Fe: Slope = 34205.2 cts/(mg/L), R^2 = 1.00000, LOD = 0.001 mg/L, LOQ = 0.002 mg/L


C:\Users\mjram\python_graphing\modules\characterization\icp.py:186: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
C:\Users\mjram\python_graphing\modules\characterization\icp.py:186: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_24388\4019927412.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Catalyst Leaching Durability Study
Assess Ni and Fe metal dissolution from 2.0 mg catalyst in 50 mL 1.0 M KOH electrolyte during 100 h OER.

In [3]:
leach_path = root_path / 'data' / 'examples' / 'characterization' / 'icp' / 'catalyst_leaching_study.csv'
leach_data = parse_icp_file(leach_path)
df_l = leach_data.dataframe
df_l_ni = df_l[df_l['element'] == 'Ni']

pct_list = []
samples = df_l_ni['sample_id'].tolist()
for _, row in df_l_ni.iterrows():
    l_res = calculate_catalyst_leaching(
        measured_concentration_mg_L=float(row['concentration']),
        electrolyte_volume_mL=50.0,
        catalyst_mass_mg=2.0,
        initial_metal_wt_fraction=0.20,
        dilution_factor=float(row.get('dilution_factor', 1.0)),
    )
    pct_list.append(l_res['percentage_metal_leached'])
    print(f'{row["sample_id"]}: {l_res["percentage_metal_leached"]:.2f}% leached ({l_res["mass_normalized_leaching_ug_per_mg"]:.1f} ug/mg_cat)')

fig, ax = plot_icp_leaching_comparison(samples, pct_list, element='Nickel', title='Catalytic Ni Leaching During OER')
plt.show()

Pristine_0h: 0.15% leached (0.3 ug/mg_cat)
OER_10h: 3.62% leached (7.2 ug/mg_cat)
OER_50h: 10.50% leached (21.0 ug/mg_cat)
OER_100h: 22.25% leached (44.5 ug/mg_cat)


C:\Users\mjram\python_graphing\modules\characterization\icp.py:239: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_24388\2319590503.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
